# Repro Paper 3: *MultiPhishGuard: An Explainable and Adaptive Multi-Agent LLM System for Phishing Email Detection*
**Venue:** arXiv:2505.23803v2 [cs.CR] (revised 23 May 2026; v1 submitted 26 May 2025)
**Authors:** Yinuo Xue, Eric Spero, Meng Wai Woo, Wei Gao, Giovanni Russello (The University of Auckland)
**Objective:** Independent experimental verification of the **multi-agent role-decomposition effect on false alarms** via a static-weight triad ablation.

> **Citation note (re-verified 2026-11-09 against the arXiv abs page):** the v2 title is
> *"An Explainable and Adaptive Multi-Agent LLM System..."* and the author list has **five**
> names, including Meng Wai Woo and Wei Gao. Earlier internal documents recorded a different
> title and four authors; that was wrong. Provenance log:
> `docs/citation_provenance/M043_MultiPhishGuard.md`.

### Primary Literature Numbers (arXiv v2, Table 1):
| Model / System | Accuracy | Precision | Recall | F1 | FPR | FNR |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| Single Agent (Zero-shot) | 84.42 | 61.31 | 99.39 | 75.84 | 20.47 | 0.61 |
| Chain-of-Thought (CoT) | 86.60 | 64.93 | 99.08 | 78.45 | 17.47 | 0.92 |
| RoBERTa-base (Supervised) | 95.73 | 86.20 | 98.67 | 91.89 | 5.13 | 1.63 |
| MultiPhishGuard (Full) | 97.89 | 92.26 | 99.80 | 95.88 | 2.73 | 0.20 |

> All values are percentages. Section 4.4 confirms the static-weight variant uses
> 0.3 (text) + 0.4 (URL) + 0.3 (metadata). The paper's Table 5 FPR for that variant is **not**
> quoted here because the value has not been re-verified against the PDF page; the goal of this
> notebook is to test whether role decomposition lowers FPR relative to single-agent CoT.

### Scope, corpus and honest limitations
- **Agents:** the full system has **five** agents (text, URL, metadata, explanation simplifier,
  adversarial) with **PPO-learned dynamic weighting** (abstract + Section 3.1). This notebook
  replicates the **three *analysis* agents only**, combined with the paper's **static** weights
  (Section 4.4) instead of PPO. Do not describe the paper as "a 3-agent system".
- **Corpus:** a public phishing/spam email corpus with real sender fields
  (`puyang2025/seven-phishing-email-datasets`, sources include Enron, CEAS-08, Assassin,
  TREC-05/06/07). This is a **construct-validity deviation**: the original benchmark uses
  Nazario phishing emails, whereas the public corpus labels spam and phishing jointly. Report it.
- **Sample:** a frozen stratified slice of 200 emails (50 phishing / 150 legitimate) drawn from
  the corpus `test` split with a fixed seed, written to disk so every re-run scores identical rows.
- **Replication nature:** `gpt-4o` => Direct replication. Any other model => Conceptual /
  Cross-model replication, and must be labelled as such.
- **Ablation confound (Rule 10.2 of RESEARCH_RULES.md):** Arm A issues **1** LLM call per email,
  Arm B issues **3**. Any FPR gain is therefore confounded with token budget and call count. Both
  are logged per sample, and the claim must be restricted to an architectural observation unless a
  matched-budget control (3 calls of the same CoT agent) is also run.

In [21]:
# Cell 1: Environment, Provider Configuration & Model Guard
!pip install -q litellm datasets scikit-learn pandas numpy scipy

import hashlib
import json
import os
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
os.environ["LLM_MODEL"] = "gemini/gemini-3.1-flash-lite"

import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as sps
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
)

# --- AGENTS.md guard: Gemini models below 3.5 are forbidden. ---
FORBIDDEN_GEMINI = re.compile(r"gemini[-_](1\.5|2\.0|2\.5)", re.IGNORECASE)

def assert_model_allowed(model_name: str) -> None:
    if FORBIDDEN_GEMINI.search(model_name):
        raise ValueError(
            f"Model '{model_name}' violates the AGENTS.md Gemini floor "
            "(< gemini-3.5-flash is forbidden). Pick gemini-3.5-flash or higher."
        )

LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o")
assert_model_allowed(LLM_MODEL)

# gpt-4o => direct replication; anything else => cross-model replication.
IS_DIRECT_REPLICATION = LLM_MODEL.strip().lower() == "gpt-4o"

# REPRO_SIMULATE=1 runs a deterministic rule-based stand-in used ONLY to smoke-test the
# pipeline offline. Its outputs are explicitly labelled non-scientific and must never be
# reported as a result.
SIMULATE = os.environ.get("REPRO_SIMULATE", "0") == "1"

API_KEY_PRESENT = any(
    os.environ.get(k)
    for k in ("OPENAI_API_KEY", "GEMINI_API_KEY", "GOOGLE_API_KEY", "OPENROUTER_API_KEY")
)

TELEMETRY_PATH = Path("Repro_Paper3_telemetry.jsonl")
BENCHMARK_PATH = Path("Repro_Paper3_Benchmark_N200_Seed42.csv")

print(f"Active LLM: {LLM_MODEL}")
print(f"Replication mode: {'DIRECT 1:1 REPLICATION' if IS_DIRECT_REPLICATION else 'CONCEPTUAL / CROSS-MODEL'}")
print(f"API key present: {API_KEY_PRESENT} | Simulation mode: {SIMULATE}")
if not API_KEY_PRESENT and not SIMULATE:
    print("\nNo API key found. Set one of OPENAI_API_KEY / GEMINI_API_KEY / "
          "OPENROUTER_API_KEY, or export REPRO_SIMULATE=1 for a pipeline smoke test only.")

Active LLM: gemini/gemini-3.1-flash-lite
Replication mode: CONCEPTUAL / CROSS-MODEL
API key present: True | Simulation mode: False


In [4]:
# Cell 2: Real Benchmark Corpus -> Frozen Stratified Slice (N=200)
from datasets import load_dataset

CORPUS_REPO = "puyang2025/seven-phishing-email-datasets"
CORPUS_SPLIT = "test"
N_PHISHING, N_LEGIT = 50, 150          # 1:3 ratio, as in the paper's corpus description
SLICE_SEED = 42

URL_RE = re.compile(r"(?:https?://|www\.)[^\s<>\"')]+", re.IGNORECASE)

def _clean(text, limit=6000):
    text = re.sub(r"[ \t]+", " ", str(text))
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text[:limit]

def _first_url(text):
    m = URL_RE.search(str(text))
    return m.group(0).rstrip(".,;)") if m else ""

def build_frozen_slice(force_rebuild=False) -> pd.DataFrame:
    if BENCHMARK_PATH.exists() and not force_rebuild:
        frozen = pd.read_csv(BENCHMARK_PATH)
        digest = hashlib.sha256(
            BENCHMARK_PATH.read_bytes()
        ).hexdigest()[:16]
        print(f"Reusing frozen benchmark {BENCHMARK_PATH} "
              f"(N={len(frozen)}, sha256:{digest}) - every re-run scores identical rows.")
        return frozen

    ds = load_dataset(CORPUS_REPO, split=CORPUS_SPLIT)
    frame = ds.to_pandas()
    frame["text"] = frame["text"].map(_clean)
    frame["sender"] = frame.get("sender", pd.Series([""] * len(frame))).fillna("")
    frame["url"] = frame["text"].map(_first_url)

    # Deduplicate on normalised text: leakage between the two arms would be fatal.
    frame["_norm"] = frame["text"].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
    frame = frame[frame["_norm"].str.len() > 40]
    frame = frame.drop_duplicates(subset=["_norm"]).reset_index(drop=True)

    phish = frame[frame["label"] == 1]
    legit = frame[frame["label"] == 0]
    if len(phish) < N_PHISHING or len(legit) < N_LEGIT:
        raise RuntimeError(f"Insufficient pool: phish={len(phish)}, legit={len(legit)}")

    sampled = pd.concat([
        phish.sample(n=N_PHISHING, random_state=SLICE_SEED),
        legit.sample(n=N_LEGIT, random_state=SLICE_SEED),
    ]).sample(frac=1.0, random_state=SLICE_SEED).reset_index(drop=True)

    sampled = sampled.reset_index().rename(columns={"index": "sample_id"})
    sampled["metadata_available"] = (sampled["sender"].astype(str).str.len() > 0).astype(int)
    keep = ["sample_id", "text", "url", "sender", "label", "metadata_available", "dataset_name"]
    out = sampled[[c for c in keep if c in sampled.columns]]
    out.to_csv(BENCHMARK_PATH, index=False)
    print(f"Frozen slice written: {BENCHMARK_PATH} (N={len(out)})")
    return out

eval_df = build_frozen_slice()

print(f"\nN={len(eval_df)} | phishing={int(eval_df['label'].sum())} | "
      f"legitimate={int((1 - eval_df['label']).sum())}")
print(f"URL extracted: {int((eval_df['url'].str.len() > 0).sum())}/{len(eval_df)}")
print(f"Sender present: {int(eval_df['metadata_available'].sum())}/{len(eval_df)}")
if "dataset_name" in eval_df.columns:
    print("Source corpora:", eval_df["dataset_name"].value_counts().to_dict())
dup = eval_df["text"].duplicated().sum()
assert dup == 0, f"{dup} duplicated texts in the frozen slice"
print("No duplicated texts in the slice.")

README.md:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  184MB            

train.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

test.parquet: downloading bytes:           |  0.00B            

eval.parquet: reconstructing file:   0%|          |  0.00B / 22.7MB            

eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/162413 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40604 [00:00<?, ? examples/s]

Frozen slice written: Repro_Paper3_Benchmark_N200_Seed42.csv (N=200)

N=200 | phishing=50 | legitimate=150
URL extracted: 86/200
Sender present: 167/200
Source corpora: {'TREC-07': 62, 'TREC-05': 55, 'CEAS-08': 33, 'Enron': 29, 'TREC-06': 13, 'Assassin': 4, 'Ling': 4}
No duplicated texts in the slice.


In [8]:
# Cell 3: Prompt Engineering - Arm A (Single CoT) vs Arm B (3 Voting Agents)
# Missing metadata is handled explicitly: the metadata agent is told to return a neutral
# 0.5 when no sender/header is supplied, so the 0.3 weight stays a documented prior
# rather than a silently fabricated signal.

COT_SYSTEM_PROMPT = """You are an expert Cyber Security Analyst evaluating incoming emails for phishing.
Analyze the email content, URL, and sender metadata step-by-step.
Return ONLY valid JSON in this exact structure:
{
  "chain_of_thought": "detailed reasoning over text, URL and sender",
  "suspicion_score": <float 0.0-1.0>,
  "is_phishing": <true|false>
}"""

TEXT_AGENT_PROMPT = """You are the Text Specialist Agent of MultiPhishGuard.
Analyze linguistic features: psychological pressure, urgency, authority pretexting, coercion,
financial lures, requests for credentials.
Return ONLY valid JSON:
{"text_risk_score": <float 0.0-1.0>, "rationale": "concise text analysis"}"""

URL_AGENT_PROMPT = """You are the URL Specialist Agent of MultiPhishGuard.
Analyze embedded hyperlinks: domain legitimacy, typosquatting, suspicious subdomains,
IP-literal hosts, risky TLDs, URL shorteners.
If no URL is present, return 0.2 (absence of a link is weak evidence of legitimacy).
Return ONLY valid JSON:
{"url_risk_score": <float 0.0-1.0>, "rationale": "concise URL analysis"}"""

METADATA_AGENT_PROMPT = """You are the Metadata Specialist Agent of MultiPhishGuard.
Analyze the sender display name vs. the actual address, domain consistency, free-mail use for
corporate claims, and spoofing indicators.
If the sender field is UNKNOWN, return exactly 0.5 with rationale "no metadata supplied".
Return ONLY valid JSON:
{"metadata_risk_score": <float 0.0-1.0>, "rationale": "concise metadata analysis"}"""

WEIGHTS = {"text": 0.3, "url": 0.4, "metadata": 0.3}
DECISION_THRESHOLD = 0.5

def _field(value, default):
    # pandas turns empty CSV cells into NaN, and NaN is truthy - so `value or default`
    # would leak the string "nan" to the agents. Normalise explicitly.
    if value is None or pd.isna(value):
        return default
    text = str(value).strip()
    return text if text and text.lower() != "nan" else default

def build_email_block(row) -> str:
    sender = _field(row.get("sender"), "UNKNOWN")
    url = _field(row.get("url"), "NONE")
    return (f"Sender: {sender}\n"
            f"Embedded URL: {url}\n"
            f"Subject/Body:\n{row['text']}")

print("Prompt contracts established for Arm A (CoT) and Arm B (static-weight triad).")
print(f"Triage weights: {WEIGHTS} | decision threshold: {DECISION_THRESHOLD}")

Prompt contracts established for Arm A (CoT) and Arm B (static-weight triad).
Triage weights: {'text': 0.3, 'url': 0.4, 'metadata': 0.3} | decision threshold: 0.5


In [22]:
# Cell 4: Execution Harness - litellm, retries, JSON repair, exact call accounting
import litellm
from litellm import completion

import logging

litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

litellm.drop_params = True
MAX_RETRIES = 4
RETRY_BASE_SECONDS = 2.0
TEMPERATURE = 0.0
MAX_TOKENS = 700

def _extract_json(raw: str) -> dict:
    if not raw:
        raise ValueError("empty LLM response")
    text = raw.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                return json.loads(text[start:i + 1])
    raise ValueError(f"no JSON object found in response: {raw[:200]!r}")

USAGE = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0,
         "cost_unavailable": 0}

def _accumulate_usage(resp) -> None:
    usage = getattr(resp, "usage", None)
    if usage is not None:
        USAGE["prompt_tokens"] += int(getattr(usage, "prompt_tokens", 0) or 0)
        USAGE["completion_tokens"] += int(getattr(usage, "completion_tokens", 0) or 0)
    try:
        from litellm import completion_cost
        USAGE["cost_usd"] += float(completion_cost(completion_response=resp) or 0.0)
    except Exception:  # noqa: BLE001 - price table may not know the model
        USAGE["cost_unavailable"] += 1

def _simulated_response(system_prompt: str, content: str) -> dict:
    # Offline stand-in for pipeline smoke tests. NOT a scientific result.
    low = content.lower()
    phish_cues = any(w in low for w in (
        "verify", "urgent", "suspended", "refund", "unauthorized", "winner",
        "click here", "act now", "confidential", "investment", "million",
    ))
    bad_domain = bool(re.search(r"(\b\d{1,3}(\.\d{1,3}){3}\b|\.(ru|top|xyz|biz|info)/|"
                                r"-secure|account-verify|login-)", low))
    if "Text Specialist" in system_prompt:
        return {"text_risk_score": 0.85 if phish_cues else 0.15, "rationale": "simulated"}
    if "URL Specialist" in system_prompt:
        return {"url_risk_score": 0.95 if bad_domain else 0.15, "rationale": "simulated"}
    if "Metadata Specialist" in system_prompt:
        if "Sender: UNKNOWN" in content:
            return {"metadata_risk_score": 0.5, "rationale": "no metadata supplied"}
        spoofy = bool(re.search(r"@(?!.*\.(edu|gov)$)[a-z0-9.-]+\.(com|net|org)$", low))
        return {"metadata_risk_score": 0.75 if spoofy else 0.25, "rationale": "simulated"}
    score = 0.75 if (phish_cues or bad_domain) else 0.2
    return {"chain_of_thought": "simulated", "suspicion_score": score,
            "is_phishing": score >= 0.5}

MIN_REQUEST_INTERVAL = 4.5  # giây giữa hai lần bắt đầu gọi API
_next_request_at = 0.0

def wait_for_api_slot():
    global _next_request_at
    wait = _next_request_at - time.monotonic()
    if wait > 0:
        time.sleep(wait)
    _next_request_at = time.monotonic() + MIN_REQUEST_INTERVAL

def call_llm(system_prompt: str, user_content: str) -> tuple[dict, float, int]:
    """Returns (parsed_json, latency_ms, n_api_calls). Raises on unrecoverable failure."""
    if SIMULATE:
        t0 = time.perf_counter()
        payload = _simulated_response(system_prompt, user_content)
        USAGE["calls"] += 1
        return payload, (time.perf_counter() - t0) * 1000.0, 1

    if not API_KEY_PRESENT:
        raise RuntimeError(
            "No LLM API key configured. Set OPENAI_API_KEY / GEMINI_API_KEY / "
            "OPENROUTER_API_KEY, or set REPRO_SIMULATE=1 for a smoke test only."
        )

    last_error = None
    for attempt in range(MAX_RETRIES):
        calls_this_attempt = attempt + 1
        # wait_for_api_slot()
        t0 = time.perf_counter()
        try:
            resp = completion(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                response_format={"type": "json_object"},
            )
            USAGE["calls"] += 1
            _accumulate_usage(resp)
            latency_ms = (time.perf_counter() - t0) * 1000.0
            payload = _extract_json(resp.choices[0].message.content)
            return payload, latency_ms, calls_this_attempt
        except Exception as exc:  # noqa: BLE001 - rate limits, timeouts, JSON errors
            last_error = exc
            USAGE["calls"] += 1
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(RETRY_BASE_SECONDS * (2 ** attempt) + np.random.uniform(0, 0.5))
    raise RuntimeError(f"LLM call failed after {MAX_RETRIES} attempts: {last_error}")

print("Harness ready: litellm routing, exponential backoff, JSON repair, per-call accounting.")

Harness ready: litellm routing, exponential backoff, JSON repair, per-call accounting.


In [23]:
# Cell 5: Save results and resume
import hashlib
import json
import os
import time
from pathlib import Path

import logging

class HideGeminiSamplingWarning(logging.Filter):
    def filter(self, record):
        msg = record.getMessage()
        return not (
            "DeprecationWarning" in msg
            and "continue to function for Gemini 3+" in msg
        )

logging.getLogger("LiteLLM").addFilter(HideGeminiSamplingWarning())

# Giữ nguyên tên này khi chạy tiếp cùng một lượt.
# Đổi tên nếu đổi model, gateway, prompt hoặc dữ liệu.
RUN_ID = "paper3_gemini_direct_run01"

RUN_DIR = Path("/kaggle/working") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

TELEMETRY_PATH = RUN_DIR / "Repro_Paper3_telemetry.jsonl"
CONFIG_PATH = RUN_DIR / "run_config.json"

assert not SIMULATE, "Đang bật mô phỏng; chưa phải thực nghiệm thật."
assert LLM_MODEL.startswith("gemini/"), f"Model hiện tại: {LLM_MODEL}"
assert len(eval_df) == 200, "Benchmark phải có 200 email."
assert eval_df["sample_id"].is_unique, "sample_id bị trùng."
assert eval_df["label"].value_counts().to_dict() == {0: 150, 1: 50}

# Đây là thông tin ghi vào hồ sơ.
# Đường gọi API thật phải được cấu hình trong call_llm() ở Cell 4.
config = {
    "model": LLM_MODEL,
    "endpoint": "https://generativelanguage.googleapis.com",
    "simulate": SIMULATE,
    "weights": WEIGHTS,
    "threshold": DECISION_THRESHOLD,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "prompts": [
        COT_SYSTEM_PROMPT,
        TEXT_AGENT_PROMPT,
        URL_AGENT_PROMPT,
        METADATA_AGENT_PROMPT,
    ],
    "dataset_sha256": hashlib.sha256(
        eval_df.to_csv(index=False).encode("utf-8")
    ).hexdigest(),
}

# Không cho trộn dự đoán từ các cấu hình khác nhau.
if CONFIG_PATH.exists():
    saved_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    assert saved_config == config, (
        "Cấu hình hoặc dữ liệu đã thay đổi. "
        "Khôi phục cấu hình cũ hoặc đặt RUN_ID mới."
    )
else:
    assert not TELEMETRY_PATH.exists(), (
        "Có telemetry nhưng thiếu run_config.json. "
        "Không tự động gộp log này."
    )
    CONFIG_PATH.write_text(
        json.dumps(config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

eval_df.to_csv(
    RUN_DIR / "Repro_Paper3_Benchmark_N200_Seed42.csv",
    index=False,
)

def append_record(record):
    with TELEMETRY_PATH.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

def is_complete(record):
    return (
        record.get("error") is None
        and type(record.get("arm_a_pred")) is int
        and record["arm_a_pred"] in (0, 1)
        and type(record.get("arm_b_pred")) is int
        and record["arm_b_pred"] in (0, 1)
    )

def risk_score(payload, field):
    value = payload.get(field)
    if type(value) not in (int, float) or not 0 <= value <= 1:
        raise ValueError(f"Thiếu hoặc sai điểm {field}")
    return float(value)

# Đọc lại kết quả đã lưu.
latest = {}
successful = {}

if TELEMETRY_PATH.exists():
    for line_no, line in enumerate(
        TELEMETRY_PATH.read_text(encoding="utf-8").splitlines(), 1
    ):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise RuntimeError(
                f"Telemetry hỏng tại dòng {line_no}. "
                "Sao lưu và kiểm tra file trước khi tiếp tục."
            ) from exc

        sid = int(record["sample_id"])
        latest[sid] = record

        if is_complete(record):
            successful[sid] = record

expected = dict(zip(
    eval_df["sample_id"].astype(int),
    eval_df["label"].astype(int),
))

assert set(latest) <= set(expected), "Log có ID ngoài benchmark."
assert all(
    r["gold_label"] == expected[sid]
    and r["model"] == LLM_MODEL
    and r["simulate"] is False
    for sid, r in latest.items()
), "Log không khớp benchmark hoặc model."

print(f"Đã hoàn tất: {len(successful)}/{len(eval_df)}")
print(f"File lưu: {TELEMETRY_PATH}")

for position, (_, row) in enumerate(eval_df.iterrows(), 1):
    sample_id = int(row["sample_id"])

    # Không gọi lại API cho mẫu đã thành công.
    if sample_id in successful:
        continue

    record = {
        "sample_id": sample_id,
        "gold_label": int(row["label"]),
        "model": LLM_MODEL,
        "simulate": False,
        "arm_a_pred": None,
        "arm_b_pred": None,
        "error": None,
    }

    started = time.perf_counter()
    calls_before = USAGE["calls"]
    prompt_before = USAGE["prompt_tokens"]
    completion_before = USAGE["completion_tokens"]
    cost_before = USAGE["cost_usd"]
    unknown_cost_before = USAGE["cost_unavailable"]

    try:
        mail = build_email_block(row)

        cot, cot_latency, _ = call_llm(COT_SYSTEM_PROMPT, mail)
        text, text_latency, _ = call_llm(TEXT_AGENT_PROMPT, mail)
        url, url_latency, _ = call_llm(URL_AGENT_PROMPT, mail)
        meta, meta_latency, _ = call_llm(METADATA_AGENT_PROMPT, mail)

        score_a = risk_score(cot, "suspicion_score")
        if type(cot.get("is_phishing")) is not bool:
            raise ValueError("Thiếu hoặc sai kiểu is_phishing")

        s_text = risk_score(text, "text_risk_score")
        s_url = risk_score(url, "url_risk_score")
        s_meta = risk_score(meta, "metadata_risk_score")

        triad = (
            WEIGHTS["text"] * s_text
            + WEIGHTS["url"] * s_url
            + WEIGHTS["metadata"] * s_meta
        )

        record.update({
            "arm_a_pred": int(cot["is_phishing"]),
            "arm_a_score": round(score_a, 4),
            "arm_b_pred": int(triad >= DECISION_THRESHOLD),
            "text_score": round(s_text, 4),
            "url_score": round(s_url, 4),
            "meta_score": round(s_meta, 4),
            "triad_score": round(triad, 4),
            "metadata_available": int(row.get("metadata_available", 0)),
            "url_present": int(bool(str(row.get("url", "")).strip())),
            "agent_latencies_ms": [
                cot_latency, text_latency, url_latency, meta_latency
            ],
        })

    except Exception as exc:
        error = str(exc)
        # Không ghi giá trị key hiện tại vào log.
        key = os.environ.get("GEMINI_API_KEY")
        if key:
            error = error.replace(key, "[REDACTED]")
        record["error"] = error[:300]

    record.update({
        "latency_ms": round((time.perf_counter() - started) * 1000, 1),
        "api_calls": int(USAGE["calls"] - calls_before),
        "prompt_tokens": int(USAGE["prompt_tokens"] - prompt_before),
        "completion_tokens": int(
            USAGE["completion_tokens"] - completion_before
        ),
        "estimated_cost_usd": round(
            USAGE["cost_usd"] - cost_before, 6
        ),
        "cost_unavailable_calls": int(
            USAGE["cost_unavailable"] - unknown_cost_before
        ),
    })

    append_record(record)
    latest[sample_id] = record

    if is_complete(record):
        successful[sample_id] = record

    print(
        f"[{position}/{len(eval_df)}] id={sample_id} "
        f"gold={record['gold_label']} "
        f"A={record['arm_a_pred']} B={record['arm_b_pred']} "
        f"calls={record['api_calls']} "
        f"lat={record['latency_ms']}ms "
        f"completed={len(successful)}/{len(eval_df)} "
        f"error={record['error'] is not None}",
        flush=True,
    )

# Khôi phục dự đoán theo đúng thứ tự để Cell 6 tính chỉ số.
ordered = [
    successful.get(int(sid), latest.get(int(sid), {}))
    for sid in eval_df["sample_id"]
]

y_true = eval_df["label"].to_numpy(dtype=int)
arm_a_preds = [r.get("arm_a_pred") for r in ordered]
arm_b_preds = [r.get("arm_b_pred") for r in ordered]
failures = len(eval_df) - len(successful)

print(f"\nHoàn tất: {len(successful)}/{len(eval_df)}")
print(f"Còn thiếu hoặc lỗi: {failures}")

Đã hoàn tất: 113/200
File lưu: /kaggle/working/paper3_gemini_direct_run01/Repro_Paper3_telemetry.jsonl
[15/200] id=14 gold=0 A=None B=None calls=4 lat=10188.5ms completed=113/200 error=True
[41/200] id=40 gold=0 A=None B=None calls=4 lat=6377.7ms completed=113/200 error=True
[54/200] id=53 gold=0 A=None B=None calls=4 lat=18509.8ms completed=113/200 error=True
[70/200] id=69 gold=0 A=None B=None calls=4 lat=14146.7ms completed=113/200 error=True
[86/200] id=85 gold=0 A=0 B=0 calls=4 lat=11804.2ms completed=114/200 error=False
[119/200] id=118 gold=0 A=0 B=0 calls=4 lat=18398.9ms completed=115/200 error=False
[120/200] id=119 gold=1 A=1 B=1 calls=5 lat=5649.1ms completed=116/200 error=False
[121/200] id=120 gold=1 A=1 B=1 calls=7 lat=14647.3ms completed=117/200 error=False
[122/200] id=121 gold=0 A=0 B=0 calls=4 lat=4045.9ms completed=118/200 error=False
[123/200] id=122 gold=0 A=0 B=0 calls=4 lat=9382.4ms completed=119/200 error=False
[124/200] id=123 gold=0 A=0 B=0 calls=4 lat=11080.3

In [18]:
# Cell 5: Run the Comparative Ablation with Per-Sample Telemetry
# Telemetry is appended and fsynced after every sample so a crash or a rate-limit wall
# never loses completed work.
import os as _os

def _append_telemetry(record: dict) -> None:
    with open(TELEMETRY_PATH, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")
        fh.flush()
        _os.fsync(fh.fileno())

if TELEMETRY_PATH.exists():
    TELEMETRY_PATH.unlink()
print(f"Telemetry -> {TELEMETRY_PATH}")

y_true = eval_df["label"].values.astype(int)
arm_a_preds, arm_b_preds = [], []
failures = 0
run_start = time.perf_counter()

for idx, row in eval_df.iterrows():
    sample_id = int(row.get("sample_id", idx))
    mail_block = build_email_block(row)
    record = {"sample_id": sample_id, "gold_label": int(row["label"]),
              "model": LLM_MODEL, "simulate": SIMULATE}
    sample_start = time.perf_counter()
    calls_before = USAGE["calls"]

    try:
        cot = call_llm(COT_SYSTEM_PROMPT, mail_block)
        cot_payload, cot_latency, _ = cot
        score_a = float(cot_payload.get("suspicion_score", 0.5))
        pred_a = int(bool(cot_payload.get("is_phishing", score_a >= DECISION_THRESHOLD)))

        text_payload, text_latency, _ = call_llm(TEXT_AGENT_PROMPT, mail_block)
        url_payload, url_latency, _ = call_llm(URL_AGENT_PROMPT, mail_block)
        meta_payload, meta_latency, _ = call_llm(METADATA_AGENT_PROMPT, mail_block)

        s_text = float(text_payload.get("text_risk_score", 0.5))
        s_url = float(url_payload.get("url_risk_score", 0.5))
        s_meta = float(meta_payload.get("metadata_risk_score", 0.5))
        triad = (WEIGHTS["text"] * s_text + WEIGHTS["url"] * s_url
                 + WEIGHTS["metadata"] * s_meta)
        pred_b = int(triad >= DECISION_THRESHOLD)

        record.update({
            "arm_a_pred": pred_a, "arm_a_score": round(score_a, 4),
            "text_score": round(s_text, 4), "url_score": round(s_url, 4),
            "meta_score": round(s_meta, 4), "triad_score": round(triad, 4),
            "arm_b_pred": pred_b,
            "metadata_available": int(row.get("metadata_available", 0)),
            "url_present": int(bool(str(row.get("url", "")).strip())),
            "latency_ms": round((time.perf_counter() - sample_start) * 1000.0, 1),
            "agent_latencies_ms": [round(cot_latency, 1), round(text_latency, 1),
                                   round(url_latency, 1), round(meta_latency, 1)],
            "error": None,
        })
        arm_a_preds.append(pred_a)
        arm_b_preds.append(pred_b)
    except Exception as exc:  # noqa: BLE001
        failures += 1
        record.update({"arm_a_pred": None, "arm_b_pred": None, "error": str(exc)[:300]})
        arm_a_preds.append(None)
        arm_b_preds.append(None)

    record["api_calls"] = int(USAGE["calls"] - calls_before)
    record["cum_prompt_tokens"] = USAGE["prompt_tokens"]
    record["cum_completion_tokens"] = USAGE["completion_tokens"]
    record["cum_cost_usd"] = round(USAGE["cost_usd"], 6)
    _append_telemetry(record)

    if (idx + 1) % 10 == 0 or idx == 0:
        print(f"[{idx + 1}/{len(eval_df)}] id={sample_id} gold={record['gold_label']} "
              f"A={record.get('arm_a_pred')} B={record.get('arm_b_pred')} "
              f"triad={record.get('triad_score')} calls={record['api_calls']} "
              f"lat={record.get('latency_ms', 'N/A')}ms failures={failures}")

wall_seconds = time.perf_counter() - run_start
print(f"\nDone in {wall_seconds / 60:.1f} min | total API calls={USAGE['calls']} | "
      f"failures={failures}")
print(f"Tokens: prompt={USAGE['prompt_tokens']:,} completion={USAGE['completion_tokens']:,}")
if USAGE["cost_unavailable"]:
    print(f"[warn] {USAGE['cost_unavailable']} calls had no price entry; cost reported as a "
          f"lower bound / N/A.")
else:
    print(f"Estimated cost: ${USAGE['cost_usd']:.4f}")

Telemetry -> Repro_Paper3_telemetry.jsonl
[1/200] id=0 gold=0 A=0 B=0 triad=0.0 calls=5 lat=15095.4ms failures=0
[10/200] id=9 gold=1 A=1 B=1 triad=0.605 calls=4 lat=3941.4ms failures=0
[20/200] id=19 gold=0 A=0 B=0 triad=0.0 calls=4 lat=4177.4ms failures=3
[30/200] id=29 gold=0 A=0 B=0 triad=0.08 calls=5 lat=16046.1ms failures=6
[40/200] id=39 gold=0 A=0 B=0 triad=0.11 calls=10 lat=42945.6ms failures=8
[50/200] id=49 gold=0 A=0 B=0 triad=0.11 calls=6 lat=11575.9ms failures=10
[60/200] id=59 gold=0 A=None B=None triad=None calls=4 lat=N/Ams failures=13
[70/200] id=69 gold=0 A=0 B=0 triad=0.07 calls=12 lat=46169.1ms failures=15
[80/200] id=79 gold=0 A=0 B=0 triad=0.08 calls=6 lat=23997.4ms failures=16
[90/200] id=89 gold=0 A=0 B=0 triad=0.2 calls=4 lat=4466.5ms failures=18
[100/200] id=99 gold=0 A=0 B=0 triad=0.07 calls=5 lat=14412.7ms failures=19
[110/200] id=109 gold=0 A=0 B=0 triad=0.125 calls=4 lat=7275.6ms failures=19
[120/200] id=119 gold=1 A=1 B=1 triad=0.835 calls=6 lat=28499.5m

In [11]:
import os
import requests

model_id = LLM_MODEL.removeprefix("gemini/")
url = f"https://generativelanguage.googleapis.com/v1alpha/models/{model_id}:generateContent"

try:
    r = requests.post(
        url,
        headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]},
        json={"contents": [{"parts": [{"text": "Reply with only: OK"}]}]},
        timeout=(5, 30),
    )
    print("HTTP status:", r.status_code)
    if r.ok:
        print("API response:", r.json()["candidates"][0]["content"]["parts"][0]["text"])
    else:
        print(r.json().get("error", {}).get("message", "Không có thông báo lỗi."))
except requests.exceptions.Timeout:
    print("Kaggle không nhận được phản hồi từ Google trong thời gian chờ.")

HTTP status: 200
API response: OK


In [8]:
from pathlib import Path

files = [
    path
    for folder in ("/kaggle/working", "/kaggle/input")
    for path in Path(folder).rglob("Repro_Paper3_telemetry.jsonl")
]

if files:
    for path in files:
        print(path)
else:
    print("Không tìm thấy file telemetry trong phiên này.")

Không tìm thấy file telemetry trong phiên này.


In [24]:
# Cell 6: Metrics, Exact McNemar, BH Correction and Clopper-Pearson Bounds
from scipy.stats import binomtest

def clopper_pearson(k, n, alpha=0.05):
    if n == 0:
        return (float("nan"), float("nan"))
    lo = 0.0 if k == 0 else sps.beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else sps.beta.ppf(1 - alpha / 2, k + 1, n - k)
    return round(float(lo) * 100, 2), round(float(hi) * 100, 2)

def performance(y_t, y_p, name):
    tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0, 1]).ravel()
    fpr_ci = clopper_pearson(fp, fp + tn)
    rec_ci = clopper_pearson(tp, tp + fn)
    return {
        "Approach": name,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "N": int(tn + fp + fn + tp),
        "Accuracy (%)": round(accuracy_score(y_t, y_p) * 100, 2),
        "Precision (%)": round(precision_score(y_t, y_p, zero_division=0) * 100, 2),
        "Recall (%)": round(recall_score(y_t, y_p) * 100, 2),
        "F1 (%)": round(f1_score(y_t, y_p) * 100, 2),
        # Rule 10.4: FPR (per legitimate email) and FDR (per raised alarm) are distinct.
        "FPR (%)": round(fp / max(fp + tn, 1) * 100, 2),
        "FPR 95% CI": f"{fpr_ci[0]}-{fpr_ci[1]}",
        "FDR (%)": round(fp / max(fp + tp, 1) * 100, 2),
        "Recall 95% CI": f"{rec_ci[0]}-{rec_ci[1]}",
    }

valid = [i for i, (a, b) in enumerate(zip(arm_a_preds, arm_b_preds))
         if a is not None and b is not None]
if len(valid) < len(eval_df):
    print(f"[warn] {len(eval_df) - len(valid)} samples failed and are excluded; "
          f"the denominators below reflect the surviving N={len(valid)} only.")

if not valid:
    raise RuntimeError("No valid predictions - inspect the telemetry JSONL for errors.")

y_v = y_true[valid]
a_v = np.array([arm_a_preds[i] for i in valid])
b_v = np.array([arm_b_preds[i] for i in valid])

rows = [
    performance(y_v, a_v, f"Arm A: Single Agent CoT ({LLM_MODEL})"),
    performance(y_v, b_v, f"Arm B: Static-Weight Triad 0.3/0.4/0.3 ({LLM_MODEL})"),
]
results_df = pd.DataFrame(rows)
display(results_df)
# results_df.to_csv("Repro_Paper3_MultiPhishGuard_Ablation.csv", index=False)
results_df.to_csv(
    RUN_DIR / "Repro_Paper3_MultiPhishGuard_Ablation.csv",
    index=False,
)
if SIMULATE:
    print("\n*** SIMULATION MODE - these numbers are a pipeline smoke test and must NOT ***")
    print("*** be reported as experimental results. Re-run with a real API key.       ***")

# --- Exact one-sided McNemar: H1 = Arm B makes fewer errors than Arm A ---
disagree_a_right = int(np.sum((a_v == y_v) & (b_v != y_v)))
disagree_b_right = int(np.sum((a_v != y_v) & (b_v == y_v)))
n_discordant = disagree_a_right + disagree_b_right
if n_discordant == 0:
    p_exact = 1.0
    print("\nNo discordant pairs: the two arms agreed on every surviving sample "
          "(p = 1.0, McNemar is uninformative).")
else:
    p_exact = float(binomtest(disagree_b_right, n_discordant, 0.5, alternative="greater").pvalue)
    chi2 = (abs(disagree_a_right - disagree_b_right) - 1.0) ** 2 / n_discordant
    p_chi2 = float(1.0 - sps.chi2.cdf(max(chi2, 0.0), df=1))
    print(f"\nDiscordant pairs: A right/B wrong={disagree_a_right}, "
          f"A wrong/B right={disagree_b_right}")
    print(f"Exact one-sided McNemar p = {p_exact:.4f}")
    print(f"Edwards-corrected chi-square cross-check: chi2={chi2:.3f}, "
          f"two-sided p={p_chi2:.4f}")

print(f"FPR delta (B - A) = {rows[1]['FPR (%)'] - rows[0]['FPR (%)']:+.2f} pp "
      f"| Recall delta = {rows[1]['Recall (%)'] - rows[0]['Recall (%)']:+.2f} pp")
print("Report the call-count asymmetry (Arm A = 1 call, Arm B = 3 calls per email) "
      "alongside these numbers; see Rule 10.2.")

[warn] 9 samples failed and are excluded; the denominators below reflect the surviving N=191 only.


,Approach,TN,FP,FN,TP,N,Accuracy (%),Precision (%),Recall (%),F1 (%),FPR (%),FPR 95% CI,FDR (%),Recall 95% CI
0,Arm A: Single Agent CoT (gemini/gemini-3.1-fla...,139,5,2,45,191,96.34,90.00,95.74,92.78,3.47,1.14-7.92,10.00,85.46-99.48
1,Arm B: Static-Weight Triad 0.3/0.4/0.3 (gemini...,140,4,9,38,191,93.19,90.48,80.85,85.39,2.78,0.76-6.96,9.52,66.74-90.85



Discordant pairs: A right/B wrong=10, A wrong/B right=4
Exact one-sided McNemar p = 0.9713
Edwards-corrected chi-square cross-check: chi2=1.786, two-sided p=0.1814
FPR delta (B - A) = -0.69 pp | Recall delta = -14.89 pp
Report the call-count asymmetry (Arm A = 1 call, Arm B = 3 calls per email) alongside these numbers; see Rule 10.2.


In [25]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
import json
import os

output = Path("/kaggle/working")
telemetry = Path(TELEMETRY_PATH).resolve()
assert telemetry.exists(), "Không tìm thấy telemetry; chưa tạo bản sao lưu."

# Lưu bảng và benchmark cùng thư mục với telemetry.
folder = telemetry.parent
metrics = folder / "Repro_Paper3_MultiPhishGuard_Ablation.csv"
benchmark = folder / "Repro_Paper3_Benchmark_N200_Seed42.csv"
summary = folder / "run_summary.json"

results_df.to_csv(metrics, index=False)
eval_df.to_csv(benchmark, index=False)

summary.write_text(
    json.dumps({
        "model": LLM_MODEL,
        "simulation": bool(SIMULATE),
        "benchmark_samples": len(eval_df),
        "paired_valid_samples": len(valid),
        "missing_or_failed_samples": len(eval_df) - len(valid),
        "exact_one_sided_mcnemar_p": float(p_exact),
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

files = [telemetry, metrics, benchmark, summary]
config = folder / "run_config.json"
if config.exists():
    files.append(config)

archive = output / "Paper3_results_backup.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for file in files:
        z.write(file, arcname=file.name)

display(FileLink(os.path.relpath(archive, Path.cwd())))
print("Bấm link Paper3_results_backup.zip phía trên để tải về máy.")

/kaggle/working/Paper3_results_backup.zip

Bấm link Paper3_results_backup.zip phía trên để tải về máy.
